In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "dataset_csv").is_dir():
    ROOT = ROOT.parent

mutation = pd.read_csv(ROOT / "dataset_csv/genomic_data/filter_gene_gbm_mutation.csv").set_index("patient_id")
expression = pd.read_csv(ROOT / "dataset_csv/genomic_data/normalized_zscore_gbm.csv").set_index("patient_id")

# Keep patients and genes available in both tables.
common_patients = mutation.index.intersection(expression.index)
common_genes = mutation.columns.intersection(expression.columns)
mutation = mutation.loc[common_patients, common_genes]
expression = expression.loc[common_patients, common_genes]
print(f"Loaded {len(common_patients)} matched patients and {len(common_genes)} shared genes.")


Loaded 250 matched patients and 3391 shared genes.


## IDH1 mutation groups
Define two groups among the matched patients using the existing binary mutation table. Here, “IDH1 no qualified” means label `0` (no recorded mutation); variant-quality criteria cannot be established from this binary table alone.

In [ ]:
# Define PTEN mutation groups: 1 = detected, 0 = no qualified mutation.
pten_groups = mutation[["PTEN"]].copy()
pten_groups["mutation_group"] = pten_groups["PTEN"].map({
    1: "PTEN mutation detected",
    0: "PTEN no qualified",
})

pten_mutation_detected = pten_groups[pten_groups["PTEN"] == 1]
pten_no_qualified = pten_groups[pten_groups["PTEN"] == 0]

display(pten_groups)
print(pten_groups["mutation_group"].value_counts())


,PTEN,mutation_group
patient_id,,
TCGA-02-0003,0,PTEN no qualified
TCGA-02-0033,0,PTEN no qualified
TCGA-02-0047,0,PTEN no qualified
TCGA-02-0055,1,PTEN mutation detected
TCGA-02-2466,0,PTEN no qualified
...,...,...
TCGA-76-4927,1,PTEN mutation detected
TCGA-76-4928,0,PTEN no qualified
TCGA-76-4929,0,PTEN no qualified


mutation_group
PTEN no qualified         172
PTEN mutation detected     78
Name: count, dtype: int64


In [ ]:
# Define EGFR mutation groups: 1 = detected, 0 = no qualified mutation.
egfr_groups = mutation[["EGFR"]].copy()
egfr_groups["mutation_group"] = egfr_groups["EGFR"].map({
    1: "EGFR mutation detected",
    0: "EGFR no qualified",
})

egfr_mutation_detected = egfr_groups[egfr_groups["EGFR"] == 1]
egfr_no_qualified = egfr_groups[egfr_groups["EGFR"] == 0]

display(egfr_groups)
print(egfr_groups["mutation_group"].value_counts())

,EGFR,mutation_group
patient_id,,
TCGA-02-0003,1,EGFR mutation detected
TCGA-02-0033,0,EGFR no qualified
TCGA-02-0047,0,EGFR no qualified
TCGA-02-0055,0,EGFR no qualified
TCGA-02-2466,1,EGFR mutation detected
...,...,...
TCGA-76-4927,1,EGFR mutation detected
TCGA-76-4928,0,EGFR no qualified
TCGA-76-4929,0,EGFR no qualified


mutation_group
EGFR no qualified         189
EGFR mutation detected     61
Name: count, dtype: int64


In [ ]:
# Define TP53 mutation groups: 1 = detected, 0 = no qualified mutation.
tp53_groups = mutation[["TP53"]].copy()
tp53_groups["mutation_group"] = tp53_groups["TP53"].map({
    1: "TP53 mutation detected",
    0: "TP53 no qualified",
})

tp53_mutation_detected = tp53_groups[tp53_groups["TP53"] == 1]
tp53_no_qualified = tp53_groups[tp53_groups["TP53"] == 0]

display(tp53_groups)
print(tp53_groups["mutation_group"].value_counts())

,TP53,mutation_group
patient_id,,
TCGA-02-0003,1,TP53 mutation detected
TCGA-02-0033,1,TP53 mutation detected
TCGA-02-0047,0,TP53 no qualified
TCGA-02-0055,1,TP53 mutation detected
TCGA-02-2466,0,TP53 no qualified
...,...,...
TCGA-76-4927,0,TP53 no qualified
TCGA-76-4928,0,TP53 no qualified
TCGA-76-4929,1,TP53 mutation detected


mutation_group
TP53 no qualified         165
TP53 mutation detected     85
Name: count, dtype: int64


In [ ]:
# Sum expression values across matched patients and rank genes
gene_sums = mutation.sum(axis=0).sort_values(ascending=False)
display(gene_sums.rename_axis("gene").reset_index(name="expression_sum"))
print(gene_sums.index.tolist())


,gene,expression_sum
0,TP53,85
1,PTEN,78
2,EGFR,61
3,RYR2,30
4,NF1,27
...,...,...
3386,NDC80,0
3387,MRPS12,0
3388,MTHFD1,0
3389,ACOX1,0


['TP53', 'PTEN', 'EGFR', 'RYR2', 'NF1', 'ATRX', 'RB1', 'SPTA1', 'PIK3CA', 'PKHD1', 'PCLO', 'LRP1', 'COL6A3', 'PIK3CG', 'NRXN2', 'LAMA1', 'IDH1', 'MAPRE3', 'SYNE1', 'RELN', 'MYH3', 'DST', 'KEL', 'AHNAK', 'TEX15', 'ABCB4', 'TCHH', 'FLNA', 'SCN9A', 'IRS4', 'MYO15A', 'DMD', 'ADAM23', 'FBN2', 'NOS1', 'TG', 'MYH1', 'GRM8', 'EPB41L3', 'MYH2', 'MXRA5', 'SLIT2', 'F5', 'ITSN1', 'VWF', 'LTBP2', 'ITGAM', 'SEMA3C', 'ANK1', 'C3', 'SLIT3', 'SAMD9', 'FLT4', 'ITGB2', 'CENPF', 'DCC', 'ATP8B1', 'SHC1', 'TMPRSS6', 'RYR1', 'LAMA2', 'CIT', 'HUWE1', 'ESPL1', 'CALCR', 'MYH11', 'ACE', 'F8', 'CRYBG1', 'FLNB', 'FLNC', 'ACACA', 'EGF', 'DSC3', 'ARFGEF1', 'KRT13', 'NPHS1', 'PPRC1', 'SCN10A', 'LAMA3', 'XDH', 'COL5A1', 'COL2A1', 'L1CAM', 'ENPEP', 'CD209', 'IL4R', 'CIITA', 'CEP250', 'MYLK', 'STAB1', 'PDE4DIP', 'PCSK2', 'MYOM2', 'PCDH7', 'IGF1R', 'COL1A2', 'SPEG', 'WNT2', 'SMC1A', 'ADAMDEC1', 'FBLN2', 'HDAC9', 'CANX', 'BRAF', 'CAD', 'CACNA1B', 'COL6A2', 'PIK3CB', 'SLC4A1', 'COL11A1', 'EPHX1', 'THSD4', 'KIT', 'KMT2D', '

In [ ]:
print(f"Count of no mutation data: {gene_sums[gene_sums == 0].count()}")
print(f"Count of mutation data: {gene_sums[gene_sums > 0].count()}")

Count of no mutation data: 143
Count of mutation data: 3248
